# MovieLens 20M — EDA, Data Cleaning & Feature Engineering

**Project:** Movie Recommender via Neural Collaborative Filtering (NCF / NeuMF)

This notebook prepares MovieLens 20M for the project described in the PRD.

### Important dataset alignment
The provided Kaggle dataset is **MovieLens 20M**, while the PRD mentions MovieLens 25M. This notebook is intentionally written for **MovieLens 20M**. Keep the dataset version consistent across experiments, reports and the final viva.

### Goals
1. Load and validate the MovieLens files.
2. Perform systematic EDA without making unsupported assumptions.
3. Clean and validate ratings/movie metadata.
4. Measure sparsity, user activity and movie activity.
5. Re-index user/movie IDs for neural embeddings.
6. Create explicit and implicit-feedback signals.
7. Prepare negative-sampling inputs.
8. Create a leave-one-out evaluation split.
9. Save reusable processed artifacts for NCF training.

**No model training is performed here.**

## 0. Environment and configuration

The notebook supports either:
- a manually downloaded/unzipped MovieLens 20M folder, or
- a Kaggle download if the Kaggle CLI is configured.

For a local setup, place the extracted files under `data/ml-20m/`.

In [3]:
from pathlib import Path
import json
import math
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Main data folder
DATA_ROOT = Path("data")

# Extracted MovieLens 20M folder
DATA_DIR = DATA_ROOT / "ml-20m"

# Folder for processed data
OUTPUT_DIR = Path("data/processed")

DATA_ROOT.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("DATA_ROOT:", DATA_ROOT.resolve())
print("DATA_DIR:", DATA_DIR.resolve())
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


DATA_ROOT: /content/data
DATA_DIR: /content/data/ml-20m
OUTPUT_DIR: /content/data/processed


## 1. Load the core files

For the core NCF pipeline, `ratings.csv` and `movies.csv` are required.

Optional files:
- `tags.csv`
- `links.csv`
- `genome-scores.csv`
- `genome-tags.csv`

We load the optional files only if they exist.

In [4]:
import urllib.request
import zipfile

ML20M_URL = "https://files.grouplens.org/datasets/movielens/ml-20m.zip"
zip_path = DATA_ROOT / "ml-20m.zip"

if (DATA_DIR / "ratings.csv").exists() and (DATA_DIR / "movies.csv").exists():
    print("Dataset already present, skipping download.")
else:
    if not zip_path.exists():
        print("Downloading MovieLens 20M (~190 MB)...")
        urllib.request.urlretrieve(ML20M_URL, zip_path)
    print("Extracting...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_ROOT)
    print("Done.")

# Fail early with a clear message if something is still missing
for fname in ["ratings.csv", "movies.csv"]:
    assert (DATA_DIR / fname).exists(), f"Missing {DATA_DIR / fname}"
print("Files found:", sorted(p.name for p in DATA_DIR.iterdir()))

Extracting...
Done.
Files found: ['README.txt', 'genome-scores.csv', 'genome-tags.csv', 'links.csv', 'movies.csv', 'ratings.csv', 'tags.csv']


In [5]:
ratings = pd.read_csv(
    DATA_DIR / "ratings.csv",
    dtype={"userId": "int32", "movieId": "int32", "rating": "float32", "timestamp": "int64"},
)
movies = pd.read_csv(DATA_DIR / "movies.csv", dtype={"movieId": "int32"})

tags_path = DATA_DIR / "tags.csv"
tags = pd.read_csv(tags_path) if tags_path.exists() else None

print("Ratings:", ratings.shape)
print("Movies:", movies.shape)
if tags is not None:
    print("Tags:", tags.shape)

Ratings: (20000263, 4)
Movies: (27278, 3)
Tags: (465564, 4)


## 2. Basic structural EDA

Check:
- dimensions
- columns
- dtypes
- sample rows
- unique users/movies
- rating range
- timestamp range

These checks establish what is actually present before cleaning.

In [6]:
display(ratings.head())
display(movies.head())

print("\nRatings info:")
ratings.info()

print("\nMovies info:")
movies.info()

print("\nUnique users:", ratings["userId"].nunique())
print("Unique movies in ratings:", ratings["movieId"].nunique())
print("Movies in movies.csv:", movies["movieId"].nunique())
print("Rating min/max:", ratings["rating"].min(), ratings["rating"].max())
print("Rating unique values:", sorted(ratings["rating"].dropna().unique()))


,userId,movieId,rating,timestamp
0,1,2,3.5,1112486027
1,1,29,3.5,1112484676
2,1,32,3.5,1112484819
3,1,47,3.5,1112484727
4,1,50,3.5,1112484580


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



Ratings info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000263 entries, 0 to 20000262
Data columns (total 4 columns):
 #   Column     Dtype  
---  ------     -----  
 0   userId     int32  
 1   movieId    int32  
 2   rating     float32
 3   timestamp  int64  
dtypes: float32(1), int32(2), int64(1)
memory usage: 381.5 MB

Movies info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27278 entries, 0 to 27277
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  27278 non-null  int32 
 1   title    27278 non-null  object
 2   genres   27278 non-null  object
dtypes: int32(1), object(2)
memory usage: 532.9+ KB

Unique users: 138493
Unique movies in ratings: 26744
Movies in movies.csv: 27278
Rating min/max: 0.5 5.0
Rating unique values: [np.float32(0.5), np.float32(1.0), np.float32(1.5), np.float32(2.0), np.float32(2.5), np.float32(3.0), np.float32(3.5), np.float32(4.0), np.float32(4.5), np.float32(5.0)]


## 3. Missing values and duplicates

Do not delete rows automatically. First quantify the problem and inspect it.

We check:
- missing values
- exact duplicate rows
- repeated `(userId, movieId)` pairs
- orphan movie IDs

In [7]:
def missing_report(df, name):
    report = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_pct": (df.isna().mean() * 100).round(4),
        "dtype": df.dtypes.astype(str)
    }).sort_values("missing_count", ascending=False)
    print(f"--- {name} ---")
    display(report)

missing_report(ratings, "ratings")
missing_report(movies, "movies")

print("Exact duplicate rating rows:", ratings.duplicated().sum())
print("Repeated user-movie pairs:", ratings.duplicated(["userId", "movieId"]).sum())

orphan_movie_ids = set(ratings["movieId"]) - set(movies["movieId"])
print("Movie IDs in ratings but absent from movies.csv:", len(orphan_movie_ids))


--- ratings ---


,missing_count,missing_pct,dtype
userId,0,0.0,int32
movieId,0,0.0,int32
rating,0,0.0,float32
timestamp,0,0.0,int64


--- movies ---


,missing_count,missing_pct,dtype
movieId,0,0.0,int32
title,0,0.0,object
genres,0,0.0,object


Exact duplicate rating rows: 0
Repeated user-movie pairs: 0
Movie IDs in ratings but absent from movies.csv: 0


## 4. Rating distribution and interaction statistics

The project uses explicit ratings, but the main recommendation task will also use an implicit-feedback representation.

We calculate:
- count of each rating
- percentage of each rating
- average rating
- median rating
- standard deviation

In [8]:
rating_summary = (
    ratings["rating"]
    .value_counts()
    .sort_index()
    .rename_axis("rating")
    .reset_index(name="count")
)
rating_summary["percentage"] = rating_summary["count"] / len(ratings) * 100

display(rating_summary)
print("Mean rating:", round(ratings["rating"].mean(), 4))
print("Median rating:", ratings["rating"].median())
print("Rating std:", round(ratings["rating"].std(), 4))


,rating,count,percentage
0,0.5,239125,1.195609
1,1.0,680732,3.403615
2,1.5,279252,1.396242
3,2.0,1430997,7.154891
4,2.5,883398,4.416932
5,3.0,4291193,21.455683
6,3.5,2200156,11.000635
7,4.0,5561926,27.809264
8,4.5,1534824,7.674019
9,5.0,2898660,14.493109


Mean rating: 3.5255
Median rating: 3.5
Rating std: 0.9998


## 5. User activity and movie activity

These distributions are central to recommender-system analysis.

We measure:
- number of ratings per user
- number of ratings per movie
- average rating per user
- average rating per movie
- minimum/median/maximum activity

These measurements reveal sparsity, highly active users, popular movies and long-tail behavior.

In [9]:
user_stats = ratings.groupby("userId").agg(
    rating_count=("movieId", "count"),
    avg_rating=("rating", "mean"),
    rating_std=("rating", "std")
).reset_index()

movie_stats = ratings.groupby("movieId").agg(
    rating_count=("userId", "count"),
    avg_rating=("rating", "mean"),
    rating_std=("rating", "std")
).reset_index()

print("User activity:")
display(user_stats["rating_count"].describe())

print("Movie activity:")
display(movie_stats["rating_count"].describe())

print("Top 10 movies by number of ratings:")
display(
    movie_stats.sort_values("rating_count", ascending=False)
    .head(10)
    .merge(movies[["movieId", "title"]], on="movieId", how="left")
    [["movieId", "title", "rating_count", "avg_rating"]]
)


User activity:


,rating_count
count,138493.000000
mean,144.413530
std,230.267257
min,20.000000
25%,35.000000
50%,68.000000
75%,155.000000
max,9254.000000


Movie activity:


,rating_count
count,26744.000000
mean,747.841123
std,3085.818268
min,1.000000
25%,3.000000
50%,18.000000
75%,205.000000
max,67310.000000


Top 10 movies by number of ratings:


,movieId,title,rating_count,avg_rating
0,296,Pulp Fiction (1994),67310,4.174231
1,356,Forrest Gump (1994),66172,4.029000
2,318,"Shawshank Redemption, The (1994)",63366,4.446990
3,593,"Silence of the Lambs, The (1991)",63299,4.177056
4,480,Jurassic Park (1993),59715,3.664741
5,260,Star Wars: Episode IV - A New Hope (1977),54502,4.190672
6,110,Braveheart (1995),53769,4.042534
7,589,Terminator 2: Judgment Day (1991),52244,3.931954
8,2571,"Matrix, The (1999)",51334,4.187186
9,527,Schindler's List (1993),50054,4.310175


## 6. Rating confidence / popularity caveat

A movie with a very high average rating but only a few ratings is not directly comparable with a movie having thousands of ratings.

For EDA, show both:
- raw average rating
- number of ratings

For optional reporting, use a minimum-rating threshold or a Bayesian/shrinkage score rather than claiming that raw average alone means "best movie".

In [10]:
MIN_RATINGS_FOR_SUMMARY = 100

qualified_movies = movie_stats[movie_stats["rating_count"] >= MIN_RATINGS_FOR_SUMMARY].copy()

print(
    f"Movies with at least {MIN_RATINGS_FOR_SUMMARY} ratings:",
    len(qualified_movies)
)

display(
    qualified_movies.sort_values(
        ["avg_rating", "rating_count"], ascending=[False, False]
    )
    .head(20)
    .merge(movies[["movieId", "title"]], on="movieId", how="left")
    [["movieId", "title", "rating_count", "avg_rating"]]
)


Movies with at least 100 ratings: 8546


,movieId,title,rating_count,avg_rating
0,318,"Shawshank Redemption, The (1994)",63366,4.446990
1,858,"Godfather, The (1972)",41355,4.364732
2,50,"Usual Suspects, The (1995)",47006,4.334372
3,527,Schindler's List (1993),50054,4.310175
4,1221,"Godfather: Part II, The (1974)",27398,4.275640
5,2019,Seven Samurai (Shichinin no samurai) (1954),11611,4.274179
6,904,Rear Window (1954),17449,4.271334
7,7502,Band of Brothers (2001),4305,4.263182
8,912,Casablanca (1942),24349,4.258327
9,922,Sunset Blvd. (a.k.a. Sunset Boulevard) (1950),6525,4.256935


## 7. Genre processing

MovieLens stores genres as pipe-separated strings such as:

`Action|Adventure|Sci-Fi`

For the hybrid/content-based extension, convert these into:
- a list of genres
- multi-hot genre columns
- genre counts

The pure NCF model does not require these columns, but they are useful for EDA and later hybrid re-ranking.

In [11]:
movies_clean = movies.copy()

movies_clean["genres"] = movies_clean["genres"].fillna("(no genres listed)")
movies_clean["genre_list"] = movies_clean["genres"].str.split("|")

all_genres = sorted({
    genre
    for genre_list in movies_clean["genre_list"]
    for genre in genre_list
})

print("Number of unique genre labels:", len(all_genres))
print(all_genres)

genre_counts = (
    movies_clean["genre_list"]
    .explode()
    .value_counts()
    .rename_axis("genre")
    .reset_index(name="movie_count")
)

display(genre_counts)


Number of unique genre labels: 20
['(no genres listed)', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy', 'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror', 'IMAX', 'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western']


,genre,movie_count
0,Drama,13344
1,Comedy,8374
2,Thriller,4178
3,Romance,4127
4,Action,3520
5,Crime,2939
6,Horror,2611
7,Documentary,2471
8,Adventure,2329
9,Sci-Fi,1743


## 8. Extract movie year from title

Movie titles commonly contain a year such as `Toy Story (1995)`.

We extract the final four-digit year when available. Missing years remain null.

This is a metadata feature for analysis/hybrid modeling, not a required input to pure NeuMF.

In [12]:
movies_clean["release_year"] = (
    movies_clean["title"]
    .str.extract(r"\((\d{4})\)\s*$", expand=False)
    .astype("Int64")
)

print("Movies with extracted year:", movies_clean["release_year"].notna().sum())
print("Movies without extracted year:", movies_clean["release_year"].isna().sum())
display(movies_clean[["movieId", "title", "release_year"]].head(10))


Movies with extracted year: 27255
Movies without extracted year: 23


,movieId,title,release_year
0,1,Toy Story (1995),1995
1,2,Jumanji (1995),1995
2,3,Grumpier Old Men (1995),1995
3,4,Waiting to Exhale (1995),1995
4,5,Father of the Bride Part II (1995),1995
5,6,Heat (1995),1995
6,7,Sabrina (1995),1995
7,8,Tom and Huck (1995),1995
8,9,Sudden Death (1995),1995
9,10,GoldenEye (1995),1995


## 9. Timestamp conversion

MovieLens timestamps are Unix timestamps.

Keep the original timestamp for reproducibility and create a datetime column for analysis and possible temporal experiments.

In [13]:
ratings_clean = ratings.copy()

ratings_clean["timestamp"] = pd.to_numeric(ratings_clean["timestamp"], errors="coerce")
ratings_clean["rating_datetime"] = pd.to_datetime(
    ratings_clean["timestamp"], unit="s", errors="coerce", utc=True
)

print("Invalid timestamps:", ratings_clean["rating_datetime"].isna().sum())
display(ratings_clean[["userId", "movieId", "rating", "timestamp", "rating_datetime"]].head())


Invalid timestamps: 0


,userId,movieId,rating,timestamp,rating_datetime
0,1,2,3.5,1112486027,2005-04-02 23:53:47+00:00
1,1,29,3.5,1112484676,2005-04-02 23:31:16+00:00
2,1,32,3.5,1112484819,2005-04-02 23:33:39+00:00
3,1,47,3.5,1112484727,2005-04-02 23:32:07+00:00
4,1,50,3.5,1112484580,2005-04-02 23:29:40+00:00


## 10. Core data cleaning

Cleaning rules used here are deliberately conservative:
1. Keep only rows with valid user/movie/rating/timestamp fields.
2. Keep ratings within the MovieLens 0.5–5.0 scale.
3. Remove exact duplicate rows.
4. Remove rating rows whose movie does not exist in `movies.csv`.
5. Do not automatically delete low-activity users/movies; make that an explicit modeling experiment.

This prevents accidental information loss.

In [14]:
before = len(ratings_clean)

ratings_clean = ratings_clean.dropna(
    subset=["userId", "movieId", "rating", "timestamp"]
).copy()

ratings_clean = ratings_clean[
    ratings_clean["rating"].between(0.5, 5.0, inclusive="both")
].copy()

ratings_clean = ratings_clean.drop_duplicates().copy()

ratings_clean = ratings_clean[
    ratings_clean["movieId"].isin(movies_clean["movieId"])
].copy()

after = len(ratings_clean)

print("Rows before cleaning:", before)
print("Rows after cleaning :", after)
print("Rows removed        :", before - after)
print("Duplicate rows left :", ratings_clean.duplicated().sum())
print("Invalid ratings left:", (~ratings_clean["rating"].between(0.5, 5.0)).sum())


Rows before cleaning: 20000263
Rows after cleaning : 20000263
Rows removed        : 0
Duplicate rows left : 0
Invalid ratings left: 0


## 11. Optional low-activity filtering

For a large NCF experiment, you may choose a minimum number of interactions per user/movie.

**Important:** filtering changes the population and therefore changes your experiment. Record the thresholds in your experiment configuration.

The default below is intentionally configurable rather than silently applied.

In [15]:
MIN_USER_RATINGS = 5
MIN_MOVIE_RATINGS = 5

user_counts = ratings_clean.groupby("userId").size()
movie_counts = ratings_clean.groupby("movieId").size()

eligible_users = user_counts[user_counts >= MIN_USER_RATINGS].index
eligible_movies = movie_counts[movie_counts >= MIN_MOVIE_RATINGS].index

ratings_filtered = ratings_clean[
    ratings_clean["userId"].isin(eligible_users)
    & ratings_clean["movieId"].isin(eligible_movies)
].copy()

print("Before activity filtering:", ratings_clean.shape)
print("After activity filtering :", ratings_filtered.shape)
print("Users retained:", ratings_filtered["userId"].nunique())
print("Movies retained:", ratings_filtered["movieId"].nunique())


Before activity filtering: (20000263, 5)
After activity filtering : (19984024, 5)
Users retained: 138493
Movies retained: 18345


## 12. Re-index users and movies for embeddings

Raw IDs are identifiers, not continuous embedding indices.

Create:
- `user_idx` in `[0, num_users-1]`
- `movie_idx` in `[0, num_movies-1]`

Keep mapping tables so predictions can be translated back to original MovieLens IDs.

In [16]:
ratings_model = ratings_filtered.copy()

user_ids = np.sort(ratings_model["userId"].unique())
movie_ids = np.sort(ratings_model["movieId"].unique())

user2idx = {int(uid): idx for idx, uid in enumerate(user_ids)}
movie2idx = {int(mid): idx for idx, mid in enumerate(movie_ids)}

ratings_model["user_idx"] = ratings_model["userId"].map(user2idx).astype(np.int64)
ratings_model["movie_idx"] = ratings_model["movieId"].map(movie2idx).astype(np.int64)

user_mapping = pd.DataFrame({
    "userId": user_ids,
    "user_idx": np.arange(len(user_ids), dtype=np.int64)
})

movie_mapping = pd.DataFrame({
    "movieId": movie_ids,
    "movie_idx": np.arange(len(movie_ids), dtype=np.int64)
})

print("Number of users:", len(user_mapping))
print("Number of movies:", len(movie_mapping))
display(ratings_model.head())


Number of users: 138493
Number of movies: 18345


,userId,movieId,rating,timestamp,rating_datetime,user_idx,movie_idx
0,1,2,3.5,1112486027,2005-04-02 23:53:47+00:00,0,1
1,1,29,3.5,1112484676,2005-04-02 23:31:16+00:00,0,28
2,1,32,3.5,1112484819,2005-04-02 23:33:39+00:00,0,31
3,1,47,3.5,1112484727,2005-04-02 23:32:07+00:00,0,46
4,1,50,3.5,1112484580,2005-04-02 23:29:40+00:00,0,49


## 13. Explicit and implicit feedback features

Keep the original rating for rating prediction.

For the recommendation/classification view, create a configurable positive-interaction threshold.

Example:
- `rating >= 4.0` → positive interaction
- otherwise → non-positive

Do not interpret an unrated movie as automatically disliked. Unrated items will be sampled separately as candidate negatives for training.

In [17]:
POSITIVE_RATING_THRESHOLD = 4.0

ratings_model["is_positive"] = (
    ratings_model["rating"] >= POSITIVE_RATING_THRESHOLD
).astype(np.int8)

print(ratings_model["is_positive"].value_counts())
print(
    "Positive interaction rate:",
    round(ratings_model["is_positive"].mean() * 100, 3),
    "%"
)


is_positive
0    9992468
1    9991556
Name: count, dtype: int64
Positive interaction rate: 49.998 %


## 14. Negative-sampling candidate preparation

For each user, define:
- positive/observed movie set
- unseen movie set

Do not materialize every user × movie pair: the matrix would be enormous.

Instead, store the observed interaction sets and sample unseen movie IDs on demand during model training.

In [18]:
# Efficient observed-item lookup
user_seen_items = (
    ratings_model.groupby("user_idx")["movie_idx"]
    .apply(set)
    .to_dict()
)

NUM_USERS = len(user_mapping)
NUM_MOVIES = len(movie_mapping)

print("Users with interaction sets:", len(user_seen_items))
print("Total user-item matrix cells if dense:", NUM_USERS * NUM_MOVIES)
print("Observed interactions:", len(ratings_model))

# Example sampler: sample unseen items for one user.
def sample_negative_items(user_idx, n_negatives=4, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    seen = user_seen_items.get(int(user_idx), set())

    negatives = []
    # Rejection sampling avoids constructing a giant complement set.
    while len(negatives) < n_negatives:
        candidate = int(rng.integers(0, NUM_MOVIES))
        if candidate not in seen:
            negatives.append(candidate)
    return negatives

example_user = int(ratings_model["user_idx"].iloc[0])
print("Example user:", example_user)
print("Sample negatives:", sample_negative_items(example_user, 5))


Users with interaction sets: 138493
Total user-item matrix cells if dense: 2540654085
Observed interactions: 19984024
Example user: 0
Sample negatives: [1710, 10130, 13064, 1395, 15872]


## 15. Leave-one-out split

For recommendation evaluation, the PRD calls for a leave-one-out protocol.

A practical chronological version:
- sort each user's interactions by timestamp
- last interaction → test
- second-last interaction → validation
- earlier interactions → train

For the main recommendation evaluation, you should decide whether the held-out interaction is:
- any rating, or
- only a positive interaction.

The code below uses the last chronological interaction and keeps the original rating for later analysis.

In [19]:
ratings_model = ratings_model.sort_values(["user_idx", "timestamp"]).copy()

# Last interaction per user = test
test_idx = ratings_model.groupby("user_idx", sort=False).tail(1).index

remaining = ratings_model.drop(index=test_idx)

# Second-last interaction per user = validation where available
val_idx = remaining.groupby("user_idx", sort=False).tail(1).index

train_df = remaining.drop(index=val_idx).copy()
val_df = remaining.loc[val_idx].copy()
test_df = ratings_model.loc[test_idx].copy()

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("Users in train:", train_df["user_idx"].nunique())
print("Users in validation:", val_df["user_idx"].nunique())
print("Users in test:", test_df["user_idx"].nunique())


Train: (19707038, 8)
Validation: (138493, 8)
Test: (138493, 8)
Users in train: 138493
Users in validation: 138493
Users in test: 138493


## 16. Feature tables for later hybrid re-ranking

Create movie metadata features:
- movie ID
- title
- release year
- genre multi-hot columns

Also calculate popularity statistics from **training data only** to avoid leakage into evaluation.

In [20]:
# Genre multi-hot matrix
genre_dummies = (
    movies_clean["genres"]
    .str.get_dummies(sep="|")
    .astype(np.int8)
)

movies_features = pd.concat(
    [
        movies_clean[["movieId", "title", "release_year"]].reset_index(drop=True),
        genre_dummies.reset_index(drop=True)
    ],
    axis=1
)

# Training-only popularity features
train_movie_stats = train_df.groupby("movieId").agg(
    train_rating_count=("userId", "count"),
    train_avg_rating=("rating", "mean")
).reset_index()

movies_features = movies_features.merge(
    train_movie_stats,
    on="movieId",
    how="left"
)

movies_features["train_rating_count"] = (
    movies_features["train_rating_count"].fillna(0).astype(np.int32)
)

print("Movie feature table shape:", movies_features.shape)
display(movies_features.head())


Movie feature table shape: (27278, 25)


,movieId,title,release_year,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,...,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western,train_rating_count,train_avg_rating
0,1,Toy Story (1995),1995,0,0,1,1,1,1,0,...,0,0,0,0,0,0,0,0,49051,3.920256
1,2,Jumanji (1995),1995,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,21969,3.207383
2,3,Grumpier Old Men (1995),1995,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,12560,3.147850
3,4,Waiting to Exhale (1995),1995,0,0,0,0,0,1,0,...,0,0,0,1,0,0,0,0,2718,2.859087
4,5,Father of the Bride Part II (1995),1995,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,12022,3.062843


## 17. Data-quality assertions

These assertions make the preprocessing pipeline reproducible and safer.

In [21]:
assert ratings_model["user_idx"].min() >= 0
assert ratings_model["movie_idx"].min() >= 0
assert ratings_model["user_idx"].max() < NUM_USERS
assert ratings_model["movie_idx"].max() < NUM_MOVIES
assert ratings_model["rating"].between(0.5, 5.0).all()
assert train_df["timestamp"].max() <= test_df["timestamp"].max()

print("All core data-quality assertions passed.")


All core data-quality assertions passed.


## 18. Save processed artifacts

The outputs are designed for later PyTorch/FastAPI work.

Files:
- `ratings_model.csv`
- `train.csv`
- `validation.csv`
- `test.csv`
- `user_mapping.csv`
- `movie_mapping.csv`
- `movies_features.csv`
- `preprocessing_config.json`

For a 20M-row dataset, Parquet is recommended for repeated local experimentation if `pyarrow` is installed.

In [22]:
# CSV outputs
ratings_model.to_csv(OUTPUT_DIR / "ratings_model.csv", index=False)
train_df.to_csv(OUTPUT_DIR / "train.csv", index=False)
val_df.to_csv(OUTPUT_DIR / "validation.csv", index=False)
test_df.to_csv(OUTPUT_DIR / "test.csv", index=False)

user_mapping.to_csv(OUTPUT_DIR / "user_mapping.csv", index=False)
movie_mapping.to_csv(OUTPUT_DIR / "movie_mapping.csv", index=False)
movies_features.to_csv(OUTPUT_DIR / "movies_features.csv", index=False)

config = {
    "seed": SEED,
    "dataset": "MovieLens 20M",
    "min_user_ratings": MIN_USER_RATINGS,
    "min_movie_ratings": MIN_MOVIE_RATINGS,
    "positive_rating_threshold": POSITIVE_RATING_THRESHOLD,
    "split": "chronological_leave_one_out",
    "notes": "Adjust thresholds as an explicit experiment; do not silently change them."
}

with open(OUTPUT_DIR / "preprocessing_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved processed artifacts to:", OUTPUT_DIR.resolve())


Saved processed artifacts to: /content/data/processed
